[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/solutions/83_async_queue_pipeline_solution.ipynb)

# 🔴 Solution: Async Queue Pipeline

Reference solution for `async_queue_pipeline`.

In [ ]:
# Install the latest torch-judge from this repo in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/CharlesShang/TorchCode.git@master')
except ImportError:
    pass


In [ ]:
import asyncio


In [ ]:
# ✅ SOLUTION

async def async_queue_pipeline(items, worker_fn, num_workers: int = 4):
    if num_workers <= 0:
        raise ValueError("num_workers must be positive")
    items = list(items)
    queue = asyncio.Queue()
    results = [None] * len(items)
    sentinel = object()
    for idx, item in enumerate(items):
        await queue.put((idx, item))
    for _ in range(num_workers):
        await queue.put(sentinel)

    async def worker():
        while True:
            job = await queue.get()
            try:
                if job is sentinel:
                    return
                idx, item = job
                results[idx] = await worker_fn(item)
            finally:
                queue.task_done()

    tasks = [asyncio.create_task(worker()) for _ in range(num_workers)]
    join_task = asyncio.create_task(queue.join())
    try:
        pending = set(tasks) | {join_task}
        while pending:
            done, pending = await asyncio.wait(pending, return_when=asyncio.FIRST_EXCEPTION)
            for task in done:
                exc = task.exception()
                if exc is not None:
                    for other in pending:
                        other.cancel()
                    await asyncio.gather(*pending, return_exceptions=True)
                    raise exc
            if join_task in done:
                break
        await join_task
        await asyncio.gather(*tasks)
    except BaseException:
        for task in tasks:
            task.cancel()
        join_task.cancel()
        await asyncio.gather(*tasks, join_task, return_exceptions=True)
        raise
    return results


In [ ]:
# Verify

def run_async(coro):
    import asyncio
    import threading
    box = {}
    def target():
        try:
            box['value'] = asyncio.run(coro)
        except BaseException as e:
            box['error'] = e
    t = threading.Thread(target=target)
    t.start()
    t.join(timeout=5)
    if t.is_alive():
        raise TimeoutError('async test timed out')
    if 'error' in box:
        raise box['error']
    return box.get('value')

async def work(x):
    await asyncio.sleep(0.01)
    return x * x
print(run_async(async_queue_pipeline(range(5), work, num_workers=2)))


In [ ]:
# Run judge
from torch_judge import check
check('async_queue_pipeline')
